In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Alerta de ordens pendentes criação de KPI                                               #              
#   ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por#
#   email e telefone de quem tem o cadastro                                                 #
#   Origem camada silver: customer                                                          #
#   Destino camada gold: orders_penging                                                     #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela silver                              #
# 2. Criar uma tabela gold com os dados da tabela silver                                    #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
caminho_codigo = '/Workspace/Users/elisangela.soares.santos83@gmail.com/Projeto_Azure_Databricks/Codigo'
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["customers","orders"],"silver")

In [0]:
%sql
describe tmp_silver_orders

In [0]:
df_orders_pending_gold = spark.sql("""
    with pending as(
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           sum(ORD.quantity) as Quantidade
    from tmp_silver_orders as ORD
    where 1 = 1
      and UPPER(ORD.status_descricao) = 'PENDING'
      group by ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao)
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           ORD.Quantidade,
           CUS.first_name,
            CUS.email,
           CUS.phone          
    FROM pending as ORD
    INNER JOIN tmp_silver_customers as CUS
           ON ORD.customer_id = CUS.customer_id
  """)


In [0]:
gold_path = buscar_caminho("gold")
# transformando consulta em uma tabela Silver 
df_orders_pending_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{gold_path}/orders_pending")